## Python REPL Tool

In [ ]:
# pip install langchain-openai langchain-experimental langchain


In [ ]:
from langchain_experimental.tools import PythonREPLTool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

import os
from dotenv import load_dotenv
from langchain_core.tools import tool

In [ ]:
from getpass import getpass
os.environ['OPENAI_API_KEY'] = getpass('Voer je OpenAI API key in: ')

In [ ]:
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')



In [ ]:
os.environ['NO_PROXY'] = '*'

llm = ChatOpenAI(
    model=model,
    api_key=api_key,
)

In [ ]:
base_repl = PythonREPLTool()

@tool
def safe_python_repl(code: str) -> str:
    """
    Use this tool to write and execute Python code.
    Use it for calculations, simulations, data analysis, or anything that requires running code.
    Input should be valid Python code as a string.
    """
    print("\n--- CODE TO BE EXECUTED ---")
    print(code)
    print("---------------------------")
    approval = input("Approve execution? (yes/no): ").strip().lower()
    if approval == "yes":
        return base_repl.run(code)
    return "Execution rejected by user."

In [ ]:
# Tool: The latter provides a human in the loop

python_tool = PythonREPLTool()
#python_tool = safe_python_repl

# Agent

tools = [python_tool]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that can write and execute Python code.",
)

# Run
from IPython.display import Markdown, display


task = """
Simulate the motion of a ball dropped from 100 meters height under gravity (g = 9.81 m/s^2).
Write a Python function that computes height after time t.
Then calculate the height at t=0, 1, 2, 3, 4, and 5 seconds.
Additionally provide the code used
"""


response = agent.invoke({"messages": [{"role": "user", "content": task}]})
display(Markdown(response["messages"][-1].content))